In [1]:
inputs = [35,25]

In [2]:
type(inputs)

list

In [3]:
weights = [0.8, 0.1]

# **Sum Function**

In [4]:
def sum_func(inputs: list, weights: list):
    # res = 0
    # for _input, _weight in zip(inputs, weights):
    #     res += _input * _weight
    return sum(input_ * weight_ for input_, weight_ in zip(inputs, weights))

In [5]:
sum_func(inputs, weights)

30.5

# **Step Function**

In [6]:
def step_func(sum):
    return int(sum >= 1)

In [7]:
s = sum_func(inputs, weights)

In [8]:
step_func(s)

1

# **Using numpy to compute sum effectively**

In [9]:
import numpy as np

In [10]:
def sum_func(inputs, weights):
    return np.array(inputs) @ np.array(weights)

In [11]:
sum_func(inputs, weights)

np.float64(30.5)

## **Gradient Descent: First Attempt at Implementation**

In [12]:
import numpy as np

def grad(features, rows, learning_rate):
    """
    assuming features = no. of features,
    rows is in the shape [[[x1, x2, x3, ...], y], [...], [...]],
    and learning rate is... learning rate :)
    """
    weights = np.random.randn(features)
    bias = 0

    for row in rows:
        x, y = row

        prediction = weights @ x + bias
        error = prediction - y

        djdw = (error) * (x) * 2.0
        djdb = (error) * 2.0

        weights -= djdw * learning_rate
        bias -= djdb * learning_rate
    return weights, bias

In [13]:
# testing with f(x) = 3x + 5

def target_func(x):
    return 3 * x + 5

In [14]:
data = [
    (np.array([x], dtype=float), target_func(x))
    for x in range(1, 100)
]

In [15]:
print(grad(1, data, 0.1))

(array([-9.62501338e+241]), np.float64(-9.722185135805507e+239))


## **Bad results because no epochs and high learning rate**

In [16]:
import numpy as np

def grad(features, rows, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    for epoch in range(epochs):

        total_loss = 0

        for x, y in rows:

            prediction = weights @ x + bias
            error = prediction - y

            total_loss += error**2

            djdw = 2 * error * x
            djdb = 2 * error

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

        if epoch % 100 == 0:
            print("Epoch:", epoch, "MSE:", total_loss / len(rows))

    return weights, bias

In [17]:
# testing with f(x) = 3x + 5

def target_func(x):
    return 3 * x + 5

In [18]:
data = [
    (np.array([x], dtype=float), target_func(x))
    for x in range(1, 100)
]

In [19]:
print(grad(1, data, 0.0001))

Epoch: 0 MSE: 282.4012062002895
Epoch: 100 MSE: 1.7752101961896534
Epoch: 200 MSE: 1.094055706478367
Epoch: 300 MSE: 0.6742626261651199
Epoch: 400 MSE: 0.4155456494134688
Epoch: 500 MSE: 0.2560993002512572
Epoch: 600 MSE: 0.15783308448003064
Epoch: 700 MSE: 0.09727196650689107
Epoch: 800 MSE: 0.05994836570095106
Epoch: 900 MSE: 0.03694596376809948
(array([3.00431647]), np.float64(4.570521586092685))


## **Attempting to apply it to a real dataset**

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [21]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [22]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [23]:
x_train = transformations.fit_transform(x_train)

In [24]:
x_train

array([[ 0.4       , -0.6       , -0.33333333, ...,  0.        ,
         1.        ,  1.        ],
       [ 0.8       ,  0.        , -0.33333333, ...,  0.        ,
         0.        ,  2.        ],
       [-0.9       , -0.2       ,  0.66666667, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.1       ,  0.        ,  0.66666667, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        , ...,  0.        ,
         0.        ,  2.        ],
       [ 0.7       ,  0.5       , -0.33333333, ...,  0.        ,
         0.        ,  2.        ]], shape=(200000, 39))

In [25]:
x_test = transformations.transform(x_test)

In [26]:
train_data = list(zip(x_train, y_train))

In [27]:
test_data = zip(x_test, y_test)

In [28]:
weights, bias = grad(x_train.shape[1], train_data, learning_rate = 1e-6, epochs = 1000)

Epoch: 0 MSE: 7297300977.219514


KeyboardInterrupt: 

In [ ]:
predictions = x_test @ weights + bias

In [ ]:
predictions.shape

(50000,)

In [ ]:
mse = sum(((y_test - predictions) ** 2)) / x_test.shape[0]

In [ ]:
mse

56642081.339566074

# **Optimizing for larger datasets**

In [29]:
import numpy as np

def grad(features, x, y, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    x, y = x.astype(np.float32), y.astype(np.float32)
    bias = 0.0

    for epoch in range(epochs):

        prediction = x @ weights + bias
        error = prediction - y

        djdw = (2 / len(x)) * (x.T @ error)
        djdb = (2 / len(x)) * np.sum(error)

        weights -= learning_rate * djdw
        bias -= learning_rate * djdb
        if epoch % 50 == 0:
            print(epoch, np.mean(error**2))
    return weights, bias

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [31]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [32]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [33]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1,1)
).flatten()

In [34]:
x_train = transformations.fit_transform(x_train)

In [35]:
x_test = transformations.transform(x_test)

In [36]:
weights, bias = grad(x_train.shape[1], x_train, y_train_scaled, learning_rate = 0.008, epochs = 3000)

0 1.0003061033728826
50 0.7597415130949088
100 0.6141792555225409
150 0.5141197062796293
200 0.44160366958225744
250 0.386595244580908
300 0.3432995924421482
350 0.3082411710083311
400 0.2792444098521022
450 0.2548834704403628
500 0.2341793939130792
550 0.21642972052802673
600 0.20111020719953918
650 0.1878164832989307
700 0.17622826722995152
750 0.16608661197007965
800 0.15717886654171767
850 0.1493283393402759
900 0.14238691949966364
950 0.13622962471888947
1000 0.13075044975536973
1050 0.1258591250962334
1100 0.12147853450450004
1150 0.117542624330886
1200 0.11399468970457347
1250 0.1107859560144625
1300 0.10787439596082096
1350 0.1052237372651841
1400 0.10280262646342252
1450 0.10058392163636248
1500 0.0985440924174113
1550 0.09666270976426723
1600 0.09492201118381206
1650 0.09330652961514053
1700 0.09180277618167977
1750 0.09039896864239065
1800 0.089084798691937
1850 0.08785123234464573
1900 0.08669033853498327
1950 0.08559514181458613
2000 0.08455949565078594
2050 0.083577973356

In [37]:
predictions = x_test @ weights + bias   

In [38]:
predictions = y_scaler.inverse_transform(
    predictions.reshape(-1,1)
).flatten()

In [39]:
from sklearn.metrics import mean_absolute_error, r2_score

mean_absolute_error(y_test, predictions)

7377.29218878424

In [40]:
r2_score(y_test, predictions)

0.9285168877213508